# Latihan: Pengantar Pemrosesan Paralel

**Nama:** Zhafira Fatihah Oswiputri
**NIM:** 2411512029

## 1. Klasifikasi Flynn

Untuk masing-masing skenario, kategori Flynn beserta alasannya:

| No | Skenario | Kategori Flynn | Alasan |
|---|---|---|---|
| a | Operasi `arr + arr` pada NumPy array | **SIMD** (Single Instruction, Multiple Data) | Satu instruksi (penjumlahan) dieksekusi serentak terhadap banyak elemen data sekaligus. NumPy memanfaatkan *vectorized instruction* (mis. AVX/SSE) di CPU, sehingga satu operasi "tambah" diterapkan ke seluruh elemen array secara paralel dalam level instruksi, bukan satu per satu seperti perulangan biasa. |
| b | Program Python biasa (tanpa threading/multiprocessing) yang membaca file lalu memproses baris demi baris | **SISD** (Single Instruction, Single Data) | Hanya ada satu aliran instruksi yang mengeksekusi satu aliran data secara berurutan. Program membaca satu baris, memprosesnya sampai selesai, baru berpindah ke baris berikutnya — tidak ada dua instruksi atau dua data yang diproses bersamaan. |
| c | Cluster komputer, masing-masing node menjalankan simulasi cuaca berbeda untuk wilayah berbeda secara independen | **MIMD** (Multiple Instruction, Multiple Data) | Tiap node menjalankan program/parameter simulasi yang bisa berbeda-beda (instruksi berbeda) terhadap data wilayahnya masing-masing (data berbeda), dan berjalan independen tanpa saling menunggu. Ini karakteristik khas komputasi terdistribusi berbasis cluster. |


## 2. Analisis Amdahl (P = 0.85)

Menghitung speedup teoritis menggunakan fungsi `amdahl_speedup(P, N)` untuk P = 0.85 dan N = 2, 4, 8, 16, 32, 64, lalu melihat pada titik mana penambahan prosesor mulai kurang berarti.

In [1]:
def amdahl_speedup(P, N):
    return 1 / ((1 - P) + P / N)

P = 0.85
N_list = [2, 4, 8, 16, 32, 64]
speedups = [amdahl_speedup(P, N) for N in N_list]

for N, S in zip(N_list, speedups):
    print(f"N = {N:>2} -> Speedup = {S:.4f}")

batas = 1 / (1 - P)
print(f"\nBatas speedup maksimum teoritis (N -> tak hingga): {batas:.4f}")

print("\nKenaikan speedup per penggandaan N:")
for i in range(1, len(N_list)):
    delta = speedups[i] - speedups[i-1]
    print(f"N={N_list[i-1]:>2} -> N={N_list[i]:>2} : +{delta:.4f}")

N =  2 -> Speedup = 1.7391
N =  4 -> Speedup = 2.7586
N =  8 -> Speedup = 3.9024
N = 16 -> Speedup = 4.9231
N = 32 -> Speedup = 5.6637
N = 64 -> Speedup = 6.1244

Batas speedup maksimum teoritis (N -> tak hingga): 6.6667

Kenaikan speedup per penggandaan N:
N= 2 -> N= 4 : +1.0195
N= 4 -> N= 8 : +1.1438
N= 8 -> N=16 : +1.0206
N=16 -> N=32 : +0.7406
N=32 -> N=64 : +0.4607

In [2]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4.5))
plt.plot(N_list, speedups, marker='o', color='darkorange', label="Speedup Amdahl")
plt.axhline(batas, color='r', linestyle='--', label=f"Batas maksimum = {batas:.2f}")
plt.title(f"Kurva Speedup Amdahl (P = {P})")
plt.xlabel("Jumlah Prosesor (N)")
plt.ylabel("Speedup S(N)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Pada N berapa mulai terjadi *diminishing returns*?**

Melihat kolom kenaikan speedup per penggandaan N: dari N=2 ke N=8 kenaikannya masih besar (sekitar +1.0 hingga +1.1 setiap kali N digandakan). Namun mulai dari N=16 ke N=32, kenaikan menurun menjadi hanya +0.74, dan dari N=32 ke N=64 tinggal +0.46 — padahal jumlah prosesor yang ditambahkan jauh lebih banyak dari sebelumnya.

Jadi *diminishing returns* mulai terlihat jelas sekitar **N = 16–32**. Ini konsekuensi langsung dari Hukum Amdahl: karena 15% dari program (1 − P) tidak bisa diparalelkan, speedup tidak akan pernah melebihi 1/(1−P) = 6,67×, berapa pun banyaknya prosesor yang ditambahkan. Semakin besar N, kontribusi tambahan setiap prosesor baru semakin kecil karena speedup sudah semakin mendekati batas asimtotik tersebut.

## 3. Benchmark Mandiri

Fungsi CPU-bound yang saya gunakan: **Fibonacci naif rekursif** `fib_rekursif(n)`. Fungsi ini sengaja tidak dioptimasi (tanpa memoization) sehingga kompleksitasnya eksponensial O(2ⁿ), cocok untuk mendemonstrasikan beban kerja CPU-bound yang naik sangat cepat. Diuji pada 4 ukuran input berbeda, diukur dengan `time.perf_counter()` **dan** `timeit`, lalu dibandingkan.

In [3]:
import time
import timeit

def fib_rekursif(n):
    """Fibonacci naif rekursif -- CPU-bound, sengaja tanpa memoization."""
    if n <= 1:
        return n
    return fib_rekursif(n - 1) + fib_rekursif(n - 2)

ukuran_input = [20, 25, 28, 30]

print("=== Pengukuran dengan time.perf_counter() ===")
hasil_perf = []
for n in ukuran_input:
    start = time.perf_counter()
    fib_rekursif(n)
    end = time.perf_counter()
    waktu = end - start
    hasil_perf.append(waktu)
    print(f"n = {n:>3} -> {waktu:.4f} detik")

print("\n=== Pengukuran dengan timeit ===")
hasil_timeit = []
for n in ukuran_input:
    waktu = timeit.timeit(f"fib_rekursif({n})", globals=globals(), number=1)
    hasil_timeit.append(waktu)
    print(f"n = {n:>3} -> {waktu:.4f} detik")

print("\n=== Tabel Perbandingan ===")
print(f"{'n':>5} | {'perf_counter (s)':>18} | {'timeit (s)':>12} | {'selisih (s)':>12}")
for n, tp, tt in zip(ukuran_input, hasil_perf, hasil_timeit):
    print(f"{n:>5} | {tp:>18.4f} | {tt:>12.4f} | {abs(tp - tt):>12.4f}")

=== Pengukuran dengan time.perf_counter() ===
n =  20 -> 0.0007 detik
n =  25 -> 0.0078 detik
n =  28 -> 0.0362 detik
n =  30 -> 0.0887 detik

=== Pengukuran dengan timeit ===
n =  20 -> 0.0007 detik
n =  25 -> 0.0079 detik
n =  28 -> 0.0323 detik
n =  30 -> 0.0843 detik

=== Tabel Perbandingan ===
    n |   perf_counter (s) |   timeit (s) |  selisih (s)
   20 |             0.0007 |       0.0007 |       0.0000
   25 |             0.0078 |       0.0079 |       0.0001
   28 |             0.0362 |       0.0323 |       0.0039
   30 |             0.0887 |       0.0843 |       0.0044

In [4]:
plt.figure(figsize=(7,4.5))
plt.plot(ukuran_input, hasil_perf, marker='o', label='perf_counter()')
plt.plot(ukuran_input, hasil_timeit, marker='s', linestyle='--', label='timeit')
plt.title("Waktu Eksekusi fib_rekursif(n) vs Ukuran Input")
plt.xlabel("n")
plt.ylabel("Waktu Eksekusi (detik)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Catatan hasil** (angka di atas dari mesin penulis — akan sedikit berbeda di komputer lain, tapi polanya serupa):

Waktu eksekusi naik sangat cepat (hampir eksponensial) seiring n bertambah, sesuai kompleksitas O(2ⁿ) dari rekursi Fibonacci naif tanpa memoization. Hasil `time.perf_counter()` dan `timeit` secara umum konsisten, karena keduanya sama-sama memakai *monotonic clock*; selisih kecil yang muncul (terutama pada n besar) wajar terjadi karena noise sistem seperti proses lain di background dan variasi scheduling OS, bukan karena metode pengukurannya berbeda secara fundamental.

## 4. Refleksi Singkat (maks. 150 kata)

Menurut saya, Hukum Amdahl lebih tepat digunakan ketika ukuran masalah **tetap** dan tujuannya murni mempercepat penyelesaian tugas yang sudah ditentukan skalanya (*strong scaling*). Contohnya adalah mengompres satu file video berdurasi tetap dengan menambah jumlah core prosesor — video itu tidak menjadi "lebih besar" hanya karena core-nya bertambah, sehingga speedup dibatasi oleh bagian kode yang tetap serial, misalnya proses membaca dan menulis file di awal-akhir.

Sebaliknya, Hukum Gustafson lebih relevan saat ukuran masalah bisa ikut membesar bersamaan dengan bertambahnya sumber daya (*weak scaling*). Contohnya pelatihan model machine learning berskala besar: dengan lebih banyak node komputasi, yang dilakukan bukan mempercepat pelatihan model yang sama, melainkan melatih model dengan data atau parameter yang lebih besar dalam rentang waktu yang kurang lebih sama.

Jadi pemilihan hukum yang tepat tergantung apakah beban kerjanya tetap (Amdahl) atau justru ikut tumbuh seiring sumber daya yang tersedia (Gustafson).